In [ ]:
# %% [markdown]
# # Задание №4: Классификация ЭЭГ для определения эпилепсии
#
# Цель: сформировать выборку из базы CHB-MIT (≥50 фрагментов с приступом и ≥50 без),
# построить вейвлет-изображения и сохранить в папки seizure/ и non_seizure/.

# %% [markdown]
# ## 1. Установка/обновление библиотек (один раз)

# %%
# Правильное имя пакета для вейвлетов — PyWavelets
!pip uninstall numpy -y
!pip install --no-cache-dir numpy==1.26.4
!pip install scipy matplotlib PyWavelets pillow mne tqdm requests --quiet

# %% [markdown]
# ## 2. Импорт модулей

# %%
import os
import numpy as np
import matplotlib.pyplot as plt
import pywt              # После установки PyWavelets
from PIL import Image
import mne
from tqdm import tqdm
import requests
import re
from pathlib import Path

# %% [markdown]
# ## 3. Параметры

# %%
DATA_ROOT = "./chbmit_data"            # Папка для скачанных данных
IMG_SEIZURE = "./seizure"              # Папка для изображений с приступом
IMG_NON_SEIZURE = "./non_seizure"      # Папка для изображений без приступа
SEGMENT_DURATION = 5.0                 # Длина фрагмента (секунд)
IMG_SIZE = (224, 224)                  # Размер итоговых изображений
MIN_SAMPLES = 50                       # Минимальное количество фрагментов каждого класса

# %% [markdown]
# ## 4. Функции загрузки данных с PhysioNet (без RECORDS, перебором файлов)

# %%
def download_file(url, dest_path):
    """Скачать файл, если он не существует, игнорировать 404."""
    if os.path.exists(dest_path):
        return True
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    print(f"Скачивание {url} -> {dest_path}")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(dest_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    elif response.status_code == 404:
        # Файл не существует – не ошибка, а отсутствие
        print(f"Файл не найден (404): {url}")
        return False
    else:
        raise RuntimeError(f"Ошибка загрузки {url}: {response.status_code}")

def download_patient_data(patient_id, base_url="https://physionet.org/files/chbmit/1.0.0"):
    """
    Загрузить все доступные .edf файлы пациента, а также summary-файл.
    Используется перебор номеров записей.
    """
    patient_dir = os.path.join(DATA_ROOT, patient_id)
    os.makedirs(patient_dir, exist_ok=True)

    # Скачиваем summary-файл (всегда есть)
    summary_file = f"{patient_id}-summary.txt"
    summary_url = f"{base_url}/{patient_id}/{summary_file}"
    summary_path = os.path.join(patient_dir, summary_file)
    download_file(summary_url, summary_path)

    # Перебираем возможные номера записей (обычно до 10-15, но для безопасности до 30)
    downloaded = 0
    for i in range(1, 30):
        rec_name = f"{patient_id}_{i:02d}.edf"
        rec_url = f"{base_url}/{patient_id}/{rec_name}"
        rec_path = os.path.join(patient_dir, rec_name)
        if download_file(rec_url, rec_path):
            downloaded += 1
        else:
            # Если не удалось скачать и это не первый файл, возможно, дальше пусто
            if i > 2 and downloaded == 0:
                break
    print(f"Загружено {downloaded} файлов .edf для {patient_id}")
    return patient_dir

# %% [markdown]
# ## 5. Загрузка пациентов chb01 … chb10 (около 2 ГБ)

# %%
patients = [f"chb{str(i).zfill(2)}" for i in range(1, 11)]
for pid in patients:
    print(f"\nЗагрузка {pid}...")
    try:
        download_patient_data(pid)
    except Exception as e:
        print(f"Ошибка при загрузке {pid}: {e}")
print("\nВсе попытки загрузки завершены.")

# %% [markdown]
# ## 6. Парсинг summary-файлов – извлечение моментов приступов

# %%
def parse_summary(summary_path):
    if not os.path.exists(summary_path):
        return {}
    with open(summary_path, 'r') as f:
        text = f.read()
    blocks = re.split(r'File Name:\s*', text)
    seizures = {}
    for block in blocks[1:]:
        lines = block.split('\n')
        edf_file = lines[0].strip()
        seizure_starts = re.findall(r'Seizure\s+Start\s+Time:\s+([\d.]+)\s+seconds', block)
        seizure_ends   = re.findall(r'Seizure\s+End\s+Time:\s+([\d.]+)\s+seconds', block)
        intervals = [(float(s), float(e)) for s, e in zip(seizure_starts, seizure_ends)]
        if intervals:
            seizures[edf_file] = intervals
    return seizures

seizure_annotations = {}
for pid in patients:
    patient_dir = os.path.join(DATA_ROOT, pid)
    summary_path = os.path.join(patient_dir, f"{pid}-summary.txt")
    if os.path.exists(summary_path):
        ann = parse_summary(summary_path)
        for edf_rel, intervals in ann.items():
            full_path = os.path.join(patient_dir, edf_rel)
            if os.path.exists(full_path):
                seizure_annotations[full_path] = intervals

print(f"Найдено файлов с разметкой и загруженных: {len(seizure_annotations)}")
total_seizures = sum(len(intervals) for intervals in seizure_annotations.values())
print(f"Всего приступов: {total_seizures}")

# Если слишком мало, можно добавить больше пациентов, но обычно хватает.

# %% [markdown]
# ## 7. Выбор ЭЭГ-канала (первый не‑стимуляционный)

# %%
def get_eeg_channel(raw):
    for i, ch in enumerate(raw.ch_names):
        if 'STI' not in ch and 'ECG' not in ch and 'EKG' not in ch:
            return i
    return 0

# %% [markdown]
# ## 8. Функция для случайного «чистого» окна (без приступа)

# %%
def get_non_seizure_window(raw, seizure_intervals, duration_sec, fs):
    total_duration = raw.n_times / fs
    max_attempts = 50
    for _ in range(max_attempts):
        start = np.random.uniform(0, total_duration - duration_sec)
        end = start + duration_sec
        overlap = False
        for (s, e) in seizure_intervals:
            if not (end <= s or start >= e):
                overlap = True
                break
        if not overlap:
            data, _ = raw[:, int(start*fs):int(end*fs)]
            return start, data[0]
    return None, None

# %% [markdown]
# ## 9. Построение вейвлет-изображения (CWT + PIL resize)

# %%
def create_wavelet_image(signal, fs, scales=range(1, 128), img_size=IMG_SIZE):
    coefs, _ = pywt.cwt(signal, scales, 'morl', sampling_period=1/fs)
    magnitude = np.abs(coefs)
    magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-10)
    magnitude = (magnitude * 255).astype(np.uint8)
    img = Image.fromarray(magnitude).resize(img_size, Image.Resampling.LANCZOS)
    return np.array(img)

# %% [markdown]
# ## 10. Генерация выборки и сохранение изображений

# %%
os.makedirs(IMG_SEIZURE, exist_ok=True)
os.makedirs(IMG_NON_SEIZURE, exist_ok=True)

seizure_count = 0
non_seizure_count = 0

for edf_path, intervals in tqdm(seizure_annotations.items(), desc="Обработка файлов"):
    print(f"\nЧтение {edf_path}")
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        fs = raw.info['sfreq']
        ch_idx = get_eeg_channel(raw)
        ch_name = raw.ch_names[ch_idx]
        print(f"Канал: {ch_name}, fs={fs} Гц")

        data, _ = raw[:, :]
        signal = data[ch_idx]

        # ---- С приступами ----
        for (start, end) in intervals:
            if seizure_count >= MIN_SAMPLES:
                break
            window_start = start
            window_end = start + SEGMENT_DURATION
            if window_end > raw.n_times / fs:
                window_end = end
                window_start = max(0, end - SEGMENT_DURATION)
                if window_start < start:
                    continue
            s_idx = int(window_start * fs)
            e_idx = int(window_end * fs)
            segment = signal[s_idx:e_idx]
            if len(segment) < fs * SEGMENT_DURATION * 0.9:
                continue
            img = create_wavelet_image(segment, fs)
            fname = f"{Path(edf_path).stem}_seizure_{start:.2f}_{window_end:.2f}.png"
            Image.fromarray(img).save(os.path.join(IMG_SEIZURE, fname))
            seizure_count += 1
            if seizure_count % 10 == 0:
                print(f"Сохранено приступов: {seizure_count}")

        # ---- Без приступов ----
        while non_seizure_count < MIN_SAMPLES:
            start_ns, seg = get_non_seizure_window(raw, intervals, SEGMENT_DURATION, fs)
            if seg is None:
                break
            img = create_wavelet_image(seg, fs)
            fname = f"{Path(edf_path).stem}_non_seizure_{start_ns:.2f}.png"
            Image.fromarray(img).save(os.path.join(IMG_NON_SEIZURE, fname))
            non_seizure_count += 1
            if non_seizure_count % 10 == 0:
                print(f"Сохранено без приступов: {non_seizure_count}")

        if seizure_count >= MIN_SAMPLES and non_seizure_count >= MIN_SAMPLES:
            break
    except Exception as e:
        print(f"Ошибка обработки {edf_path}: {e}")

print("\n--- Результат ---")
print(f"Снимков с приступом: {seizure_count}")
print(f"Снимков без приступа: {non_seizure_count}")

# %% [markdown]
# ## 11. Просмотр нескольких примеров

# %%
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
seizure_imgs = list(Path(IMG_SEIZURE).glob("*.png"))[:4]
non_imgs = list(Path(IMG_NON_SEIZURE).glob("*.png"))[:4]

for i, p in enumerate(seizure_imgs):
    axes[0, i].imshow(plt.imread(p), cmap='gray')
    axes[0, i].set_title("Seizure")
    axes[0, i].axis('off')
for i, p in enumerate(non_imgs):
    axes[1, i].imshow(plt.imread(p), cmap='gray')
    axes[1, i].set_title("Non‑seizure")
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

# %% [markdown]
# ## Готово!
#
# - Папки `seizure` и `non_seizure` содержат не менее 50 вейвлет-изображений каждая.
# - Эти изображения можно использовать для обучения свёрточной нейронной сети.

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/19/77/538f202862b9183f54108557bfda67e17603fc560c384559e769321c9d92/numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
    --------------------------------------- 0.2/15.8 MB 6.9 MB/s eta 0:00:03
   - -------------------------------------- 0.8/15.8 MB 8.2 MB/s eta 0:00:02
   ----- ---------------------------------- 2.0/15.8 MB 15.8 MB/s eta 0:00:01
   ------------- -------------------------- 5.5/15.8 MB 32.0 MB/s eta 0:00:01
   ---------------------------- ----------- 11.5/15.8 MB 81.8 MB/s eta 0:00:01
   ---------------------------------------  15.7/15.8 MB 108.8 MB/s

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.13.1 requires keras<2.14,>=2.13.1, but you have keras 3.12.1 which is incompatible.
tensorflow-intel 2.13.1 requires numpy<=1.24.3,>=1.22, but you have numpy 1.26.4 which is incompatible.
tensorflow-intel 2.13.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.34.1 which is incompatible.
tensorflow-intel 2.13.1 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.15.0 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



Загрузка chb01...
Скачивание https://physionet.org/files/chbmit/1.0.0/chb01/chb01-summary.txt -> ./chbmit_data\chb01\chb01-summary.txt
Скачивание https://physionet.org/files/chbmit/1.0.0/chb01/chb01_01.edf -> ./chbmit_data\chb01\chb01_01.edf
